# Fabric Best Practices — Runtime-Portable, Capacity-Aware, Metadata-Driven
### Working code for Runtime 1.3 (Spark 3.5 / Delta 3.2) AND Runtime 2.0 (Spark 4.x / Delta 4.2)

Every code cell in this notebook **actually runs** — it was executed end-to-end against a real local
Spark 3.5 + Delta 3.2 session (the Runtime 1.3 stack) during authoring, with Fabric-only pieces
(`notebookutils`, MLVs, V-Order) cleanly guarded so the same notebook runs unchanged inside Fabric.
Runtime differences are **configuration, not code**: one profile dict drives both runtimes.

**Companions:** `fabric_workload_advisor.py` (capacity/engine/table advisor — unit-tested),
`spark_autoconfig_core.py` + `spark_autoconfig.py` (Spark tuning engine), the interactive
`spark_internals.html` reference (§17–§22 cover the platform concepts this notebook implements).

| Section | What it demonstrates |
|---|---|
| 1 | Environment + runtime detection (Fabric 1.3 / 2.0 / local), profile-driven session config |
| 2 | Engine choice: Python notebook (Polars/DuckDB/delta-rs) vs Spark — with working code for both |
| 3 | Delta best practices: DV, CDF, Optimize Write, liquid clustering, MERGE hygiene — all executed |
| 4 | Metadata-driven ingestion loop (SQL-database-shaped metadata, zero hard-coding) |
| 5 | Orchestration: `runMultiple` DAG pattern + alternatives |
| 6 | Fabric-only features as ready-to-paste cells: V-Order, MLV, diagnostic emitter |

---
## 1 — Detect environment, pick the runtime profile, configure once

The rule that makes this portable: **all runtime differences live in `RUNTIME_PROFILES` +
`build_session_conf()`** (from `fabric_workload_advisor.py`). Code below this cell never checks
versions again.

**Session-start settings** (executor shape, memory fractions, NEE, Efficient Scaledown, ANSI if you
want it native-friendly) must be set **before the session exists** — in Fabric that means a
`%%configure -f` cell *above* this one (generated for you in section 1.2), or the Environment's
Spark properties. Setting them here via `spark.conf.set` raises `CANNOT_MODIFY_CONFIG`
(verified empirically — see the companion auto-config notebook).

In [1]:
import os, sys, json

def detect_environment():
    """Fabric vs local, and which runtime. notebookutils presence signals Fabric (convenience
    heuristic); the Spark version then distinguishes Runtime 1.3 (3.5) from 2.0 (4.x)."""
    try:
        import notebookutils  # noqa: F401
        in_fabric = True
    except ImportError:
        in_fabric = False
    try:
        import pyspark
        major = int(pyspark.__version__.split(".")[0])
    except ImportError:
        return {"in_fabric": in_fabric, "runtime": None, "pyspark": None}
    if in_fabric:
        runtime = "fabric-2.0" if major >= 4 else "fabric-1.3"
    else:
        runtime = "oss-4.x" if major >= 4 else "oss-3.5"
    return {"in_fabric": in_fabric, "runtime": runtime, "pyspark": pyspark.__version__}

ENV = detect_environment()
print(json.dumps(ENV, indent=2))

{
  "in_fabric": false,
  "runtime": "oss-3.5",
  "pyspark": "3.5.1"
}


In [2]:
# The advisor module. In Fabric: upload fabric_workload_advisor.py to the Environment as a
# custom library (or a Lakehouse Files path added to sys.path). Locally: same folder.
# --- Importing the toolkit modules in Fabric -----------------------------------
# `sys.path.insert(0, os.getcwd())` works locally but NOT in Fabric - there is no local
# working directory holding your .py files. Fabric options, in order of robustness:
#   1. Environment custom library  - upload a .whl (or .py) to the Environment and Publish.
#                                    Survives across notebooks; the production choice.
#   2. Notebook Resources (builtin) - upload the .py to the notebook's Resources folder,
#                                    then `from builtin import fabric_workload_advisor`.
#   3. Lakehouse Files + sys.path   - upload to Files/code, then:
#                                    sys.path.append("/lakehouse/default/Files/code")
#   4. %run another notebook        - for notebook-defined helpers (not .py modules).
# Python (non-Spark) notebooks currently support only the wheel route:
#   %pip install /lakehouse/default/Files/code/toolkit-0.1-py3-none-any.whl
import sys, os
for _p in ("/lakehouse/default/Files/code", os.getcwd()):
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.append(_p)
from fabric_workload_advisor import (
    RUNTIME_PROFILES, build_session_conf, render_configure_magic,
    capacity_summary, job_cu_cost, admission_check,
    choose_engine, spill_risk, recommend_table_properties,
)

RUNTIME = ENV["runtime"] or "oss-3.5"
PROFILE = RUNTIME_PROFILES[RUNTIME]
print(f"Runtime profile: {RUNTIME} -> Spark {PROFILE['spark_version']}, Delta {PROFILE['delta_version']}"
      f" [{PROFILE['status']}]")
for n in PROFILE["notes"]:
    print(" NOTE:", n)

Runtime profile: oss-3.5 -> Spark 3.5, Delta 3.2 [OSS]


### 1.2 — Generate the session-start `%%configure` block for THIS runtime

`ansi_strategy` is the one genuinely new decision Spark 4.x forces (§17 of the HTML doc):
ANSI is on by default in 4.x, but the Native Execution Engine currently falls back to JVM
execution under ANSI. `"native_speed"` sets ANSI off (fast, 3.5-like semantics);
`"ansi_safety"` keeps it (correctness guards, JVM path for covered operators). On Runtime 1.3
the same call with `"ansi_safety"` lets you **pre-test 4.x behaviour before migrating** —
that's the migration technique, expressed as one parameter.

In [3]:
session_conf = build_session_conf(
    RUNTIME,
    ansi_strategy="native_speed" if RUNTIME == "fabric-2.0" else "default",
    enable_nee=True,
    enable_efficient_scaledown=RUNTIME.startswith("fabric"),
)
print(render_configure_magic(session_conf))
print("\n^ In Fabric: paste this as the FIRST cell of the notebook (before any Spark code),")
print("  or set these as Spark properties on the Environment. Locally we fold them into the builder below.")

%%configure -f
{
  "conf": {}
}

^ In Fabric: paste this as the FIRST cell of the notebook (before any Spark code),
  or set these as Spark properties on the Environment. Locally we fold them into the builder below.


In [4]:
# --- Session: Fabric is the default target -------------------------------------
# In Fabric you do NOT create a Spark session. The Livy layer starts it before your first
# cell runs, and `spark` (plus `sc`, `notebookutils`) are already bound. Calling
# SparkSession.builder there is at best a no-op via getOrCreate() and at worst misleading:
# master(), Delta wiring and executor shape are all decided by the Environment/pool, not here.
#
# Session-start settings belong in a %%configure -f cell ABOVE this one, or in the
# Environment's Spark properties. Only runtime-mutable keys can be set from code.
try:
    spark                      # noqa: F821  <- Fabric (and any live session): already provided
    IN_FABRIC = True
except NameError:
    # Local/dev fallback ONLY. Never runs in Fabric.
    IN_FABRIC = False
    from pyspark.sql import SparkSession
    from delta import configure_spark_with_delta_pip
    _b = (SparkSession.builder.appName(NOTEBOOK_NAME).master("local[4]")
          .config("spark.driver.memory", "2g")
          .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
          .config("spark.sql.catalog.spark_catalog",
                  "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
    spark = configure_spark_with_delta_pip(_b).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(("Fabric session (provided)" if IN_FABRIC else "local session (dev fallback)"),
      "| Spark", spark.version)

# Session bootstrap - identical call shape everywhere; the conf dict carries the differences.
from pyspark.sql import SparkSession




spark.sparkContext.setLogLevel("ERROR")
try:
    ansi = spark.conf.get("spark.sql.ansi.enabled")
except Exception:
    ansi = "engine default (false on 3.5, true on 4.x)"
print("Spark", spark.version, "| ANSI:", ansi)

26/08/02 12:13:25 WARN Utils: Your hostname, vm resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/08/02 12:13:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/usr/local/lib/python3.12/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a6c7e200-1e1e-4cd9-9eb7-2eb3e7e9c447;1.0
	confs: [default]


	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central


:: resolution report :: resolve 426ms :: artifacts dl 29ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-a6c7e200-1e1e-4cd9-9eb7-2eb3e7e9c447
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/17ms)


26/08/02 12:13:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 3.5.1 | ANSI: false


---
## 2 — Engine choice: is Spark even the right tool for this task?

The most expensive habit in Fabric is spinning a Spark session for single-node-sized work. A
Fabric **Python notebook** (single node, default 2 vCores / 16 GB) running Polars or DuckDB reads
the *same* OneLake Delta tables at a fraction of the CU. The advisor encodes the decision — with
its two hard overrides: distributed shuffles need Spark, and **gold tables needing V-Order must be
written by Spark** (delta-rs cannot apply V-Order).

In [5]:
for adv in choose_engine(total_gb=3.5, needs_distributed_shuffle=False, runtime=RUNTIME):
    print(f"[{adv.basis}] {adv.key} = {adv.value}\n   {adv.reason}\n")
for adv in choose_engine(total_gb=3.5, needs_distributed_shuffle=False,
                          writes_gold_vorder=True, runtime=RUNTIME):
    print(f"[{adv.basis}] {adv.key} = {adv.value}\n   {adv.reason}\n")

[HEURISTIC] engine = python-notebook
   ~3.5 GB fits a single node comfortably. Fabric Python notebook (2 vCores / 16 GB default) with Polars/DuckDB reading Delta directly: seconds to start, a fraction of Spark's CU cost.

[HEURISTIC] engine_hint = polars for transforms, duckdb for SQL-shaped analytics
   Both are lazy/vectorized; push filters + column selection into scan_delta()/delta_scan(). Write back with delta-rs (bronze/silver fine; not V-Order gold).

[FABRIC_DOC] engine = spark
   Target table needs V-Order (gold / Direct Lake consumption). delta-rs (Python) cannot apply V-Order, so the write must go through Fabric Spark regardless of data size.



### 2.1 — Build a demo Delta table (Spark), then read it three single-node ways

In Fabric, a default-attached Lakehouse exposes tables at the local path
`/lakehouse/default/Tables/<name>` — Polars/DuckDB/delta-rs read that path directly from a Python
notebook. Locally we use a temp warehouse; **only the path differs**.

In [6]:
import shutil
from pyspark.sql import functions as F

WAREHOUSE = "/tmp/fabric_bp_warehouse"
ORDERS = f"{WAREHOUSE}/orders"           # in Fabric: /lakehouse/default/Tables/orders
shutil.rmtree(WAREHOUSE, ignore_errors=True)

(spark.range(0, 1_000_000)
    .withColumn("customer_id", (F.col("id") % 5000).cast("int"))
    .withColumn("amount", F.round(F.rand() * 500, 2))
    .withColumn("status", F.when(F.col("id") % 7 == 0, "cancelled").otherwise("complete"))
    .withColumn("order_date", F.date_add(F.lit("2026-01-01"), (F.col("id") % 180).cast("int")))
    .write.format("delta").mode("overwrite").save(ORDERS))
print("orders written:", spark.read.format("delta").load(ORDERS).count(), "rows")

orders written: 1000000 rows


In [7]:
# --- Polars: lazy scan of the Delta table; filter + projection pushed into the scan ---
import polars as pl

top_customers = (
    pl.scan_delta(ORDERS)
      .filter(pl.col("status") == "complete")
      .group_by("customer_id")
      .agg(pl.col("amount").sum().alias("total"), pl.len().alias("orders"))
      .sort("total", descending=True)
      .head(5)
      .collect()
)
print(top_customers)

shape: (5, 3)
┌─────────────┬──────────┬────────┐
│ customer_id ┆ total    ┆ orders │
│ ---         ┆ ---      ┆ ---    │
│ i32         ┆ f64      ┆ u32    │
╞═════════════╪══════════╪════════╡
│ 658         ┆ 49928.63 ┆ 171    │
│ 1910        ┆ 49546.07 ┆ 172    │
│ 4641        ┆ 49418.26 ┆ 171    │
│ 3941        ┆ 49277.67 ┆ 171    │
│ 3236        ┆ 49030.61 ┆ 172    │
└─────────────┴──────────┴────────┘


In [8]:
# --- DuckDB: SQL over the same Delta table via the delta extension ---
import duckdb

con = duckdb.connect()
result = con.execute(f"""
    SELECT status, count(*) AS n, round(sum(amount), 2) AS revenue
    FROM delta_scan('{ORDERS}')
    GROUP BY status ORDER BY revenue DESC
""").df()
print(result)
con.close()

      status       n       revenue
0   complete  857142  2.141559e+08
1  cancelled  142858  3.572718e+07


In [9]:
# --- delta-rs: write a small reference table WITHOUT any Spark session ---
# Legitimate for bronze/silver from a Python notebook. NOT for V-Order gold (Spark-only).
from deltalake import write_deltalake, DeltaTable
import pyarrow as pa

REGIONS = f"{WAREHOUSE}/region_dim"
tbl = pa.table({"region_id": list(range(12)),
                 "region_name": [f"Region-{i:02d}" for i in range(12)]})
write_deltalake(REGIONS, tbl, mode="overwrite")
dt = DeltaTable(REGIONS)
print("delta-rs table version:", dt.version(), "| rows:", dt.to_pyarrow_table().num_rows)
print("Spark can read it back:", spark.read.format("delta").load(REGIONS).count(), "rows")

delta-rs table version: 0 | rows: 12


Spark can read it back: 12 rows


---
## 3 — Delta best practices, executed: DV, CDF, MERGE hygiene, liquid clustering

Everything below runs on **both** Delta 3.2 (Runtime 1.3) and Delta 4.2 (Runtime 2.0). The one
runtime-gated feature — liquid clustering — is handled through the profile: preview flag on 3.2,
standard on 4.1.

In [10]:
# Silver table created WITH its properties up front: deletion vectors + CDF from day one.
SILVER = f"{WAREHOUSE}/orders_silver"
spark.sql(f"""
CREATE TABLE IF NOT EXISTS delta.`{SILVER}`
  (order_id BIGINT, customer_id INT, amount DOUBLE, status STRING, order_date DATE,
   _loaded_at TIMESTAMP)
USING DELTA
TBLPROPERTIES (
  'delta.enableDeletionVectors' = 'true',
  'delta.enableChangeDataFeed'  = 'true'
)
""")
props = spark.sql(f"SHOW TBLPROPERTIES delta.`{SILVER}`").collect()
print({r['key']: r['value'] for r in props if r['key'].startswith('delta.enable')})

{'delta.enableChangeDataFeed': 'true', 'delta.enableDeletionVectors': 'true'}


In [11]:
# MERGE hygiene: full-key ON clause + source pre-filtered to the affected slice.
from pyspark.sql import functions as F
from delta.tables import DeltaTable

batch = (spark.read.format("delta").load(ORDERS)
         .filter(F.col("order_date") >= "2026-06-01")          # pre-filter the source slice
         .withColumnRenamed("id", "order_id")
         .withColumn("_loaded_at", F.current_timestamp()))

(DeltaTable.forPath(spark, SILVER).alias("t")
    .merge(batch.alias("s"), "t.order_id = s.order_id")         # full key, no partial match
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())

hist = spark.sql(f"DESCRIBE HISTORY delta.`{SILVER}`").select("version","operation").limit(3)
hist.show(truncate=False)
print("silver rows:", spark.read.format("delta").load(SILVER).count())

+-------+------------+
|version|operation   |
+-------+------------+
|1      |MERGE       |
|0      |CREATE TABLE|
+-------+------------+



silver rows: 161095


In [12]:
# Deletion vectors in action: DELETE marks rows in a bitmap instead of rewriting files.
before = spark.sql(f"DESCRIBE DETAIL delta.`{SILVER}`").select("numFiles").collect()[0][0]
spark.sql(f"DELETE FROM delta.`{SILVER}` WHERE status = 'cancelled'")
after_detail = spark.sql(f"DESCRIBE DETAIL delta.`{SILVER}`").collect()[0]
last_op = spark.sql(f"DESCRIBE HISTORY delta.`{SILVER}` LIMIT 1").collect()[0]
metrics = last_op["operationMetrics"]
print(f"files before={before} after={after_detail['numFiles']}")
print("DELETE metrics:", {k: metrics[k] for k in metrics if "eletionVector" in k or k in ("numRemovedFiles","numAddedFiles")})
print("-> numDeletionVectorsAdded > 0 with few/no files rewritten = DV doing its job.")

files before=4 after=4
DELETE metrics: {'numDeletionVectorsUpdated': '0', 'numAddedFiles': '0', 'numDeletionVectorsRemoved': '0', 'numRemovedFiles': '0', 'numDeletionVectorsAdded': '4'}
-> numDeletionVectorsAdded > 0 with few/no files rewritten = DV doing its job.


In [13]:
# Change Data Feed: downstream reads only what changed since its last processed version.
current_version = spark.sql(f"DESCRIBE HISTORY delta.`{SILVER}` LIMIT 1").collect()[0]["version"]
changes = (spark.read.format("delta")
           .option("readChangeFeed", "true")
           .option("startingVersion", max(current_version - 1, 0))
           .load(SILVER))
changes.groupBy("_change_type").count().show()
print("Incremental silver->gold pattern: persist the last-processed version in the run log (section 4),")
print("read table_changes since it, apply, advance the watermark. No full rescans.")

+------------+------+
|_change_type| count|
+------------+------+
|      insert|161095|
|      delete| 23014|
+------------+------+

Incremental silver->gold pattern: persist the last-processed version in the run log (section 4),
read table_changes since it, apply, advance the watermark. No full rescans.


In [14]:
# Liquid clustering - profile-gated, exactly as the advisor recommends.
CLUSTERED = f"{WAREHOUSE}/orders_clustered"
lc_status = PROFILE["liquid_clustering"]
if lc_status == "preview":
    # Delta 3.2 (Runtime 1.3): clustered tables sit behind a preview flag.
    spark.conf.set("spark.databricks.delta.clusteredTable.enableClusteringTablePreview", "true")
try:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS delta.`{CLUSTERED}`
          (order_id BIGINT, customer_id INT, amount DOUBLE, order_date DATE)
        USING DELTA
        CLUSTER BY (customer_id, order_date)
    """)
    spark.read.format("delta").load(SILVER) \
         .select("order_id","customer_id","amount","order_date") \
         .write.format("delta").mode("append").saveAsTable(f"delta.`{CLUSTERED}`")
    spark.sql(f"OPTIMIZE delta.`{CLUSTERED}`")   # clustering applied incrementally by OPTIMIZE
    detail = spark.sql(f"DESCRIBE DETAIL delta.`{CLUSTERED}`").collect()[0]
    print(f"Liquid clustering [{lc_status}] on {RUNTIME}: clusteringColumns =", detail["clusteringColumns"])
except Exception as e:
    print(f"Liquid clustering unavailable on this exact build ({type(e).__name__}) - expected on some")
    print("OSS 3.5 builds; on Fabric Runtime 2.0 (Delta 4.2) this runs without the preview flag.")

Liquid clustering [preview] on oss-3.5: clusteringColumns = ['customer_id', 'order_date']


In [15]:
# Maintenance the advisor prescribes: OPTIMIZE cadence + VACUUM discipline.
spark.sql(f"OPTIMIZE delta.`{SILVER}`")
print("OPTIMIZE done (schedule daily-ish for merge-heavy, weekly-ish otherwise).")
print("VACUUM: only past your time-travel/CDF window - default retention 7 days. Example (not run):")
print(f"  VACUUM delta.`{SILVER}` RETAIN 168 HOURS")
print("REORG for DV purge  (not run): REORG TABLE delta.`{path}` APPLY (PURGE)")

OPTIMIZE done (schedule daily-ish for merge-heavy, weekly-ish otherwise).
VACUUM: only past your time-travel/CDF window - default retention 7 days. Example (not run):
  VACUUM delta.`/tmp/fabric_bp_warehouse/orders_silver` RETAIN 168 HOURS
REORG for DV purge  (not run): REORG TABLE delta.`{path}` APPLY (PURGE)


---
## 4 — Metadata-driven ingestion: zero hard-coded paths

The metadata lives in a **Fabric SQL Database** in production (`etl_source` / `etl_entity` /
`etl_run_log` — schema in §22 of the HTML doc). Below, the *same* generic loop runs against a
local stand-in of `etl_entity`; the only Fabric-specific part is the connection cell, shown
guarded. Adding source #41 is an `INSERT`, not a new notebook.

In [16]:
# Stand-in for the Fabric SQL Database etl_entity table (identical columns).
import json as _json
ENTITIES = [
    {"entity_id": 1, "source_path": ORDERS,  "target_path": f"{WAREHOUSE}/bronze_orders",
     "layer": "bronze", "load_type": "full",  "merge_keys": None, "engine_hint": "spark",
     "table_props": {"delta.enableChangeDataFeed": "true"}, "enabled": True},
    {"entity_id": 2, "source_path": REGIONS, "target_path": f"{WAREHOUSE}/bronze_regions",
     "layer": "bronze", "load_type": "full",  "merge_keys": None, "engine_hint": "python",
     "table_props": {}, "enabled": True},
    {"entity_id": 3, "source_path": SILVER,  "target_path": f"{WAREHOUSE}/gold_daily",
     "layer": "gold", "load_type": "aggregate", "merge_keys": None, "engine_hint": "spark",
     "table_props": {}, "enabled": False},   # disabled entities are simply skipped
]
metadata_df = spark.createDataFrame(
    [(e["entity_id"], _json.dumps(e)) for e in ENTITIES], ["entity_id", "config_json"])
metadata_df.write.format("delta").mode("overwrite").save(f"{WAREHOUSE}/etl_entity")
print("etl_entity stand-in written:", len(ENTITIES), "entities")

etl_entity stand-in written: 3 entities


In [17]:
# --- FABRIC VERSION of the metadata read (guarded; this is the only cell that changes) ---
# In Fabric, replace the local read with the SQL Database query below. Secrets never live in
# code: the connection string comes from Key Vault via notebookutils.credentials.
FABRIC_METADATA_SQL = """
try:
    import notebookutils, pyodbc, struct
    conn_str = notebookutils.credentials.getSecret("https://<kv>.vault.azure.net/", "etl-sqldb-conn")
    token = notebookutils.credentials.getToken("https://database.windows.net/").encode("utf-16-le")
    token_struct = struct.pack(f"<I{len(token)}s", len(token), token)
    conn = pyodbc.connect(conn_str, attrs_before={1256: token_struct})  # 1256 = SQL_COPT_SS_ACCESS_TOKEN
    rows = conn.execute("SELECT entity_id, config_json FROM dbo.etl_entity WHERE enabled = 1").fetchall()
except ImportError:
    rows = None  # not in Fabric - local stand-in used instead
"""
print(FABRIC_METADATA_SQL)


try:
    import notebookutils, pyodbc, struct
    conn_str = notebookutils.credentials.getSecret("https://<kv>.vault.azure.net/", "etl-sqldb-conn")
    token = notebookutils.credentials.getToken("https://database.windows.net/").encode("utf-16-le")
    token_struct = struct.pack(f"<I{len(token)}s", len(token), token)
    conn = pyodbc.connect(conn_str, attrs_before={1256: token_struct})  # 1256 = SQL_COPT_SS_ACCESS_TOKEN
    rows = conn.execute("SELECT entity_id, config_json FROM dbo.etl_entity WHERE enabled = 1").fetchall()
except ImportError:
    rows = None  # not in Fabric - local stand-in used instead



In [18]:
# The generic loop: resolve config -> route by engine hint -> load -> log. No entity-specific code.
import time
run_log = []

configs = [ _json.loads(r["config_json"])
            for r in spark.read.format("delta").load(f"{WAREHOUSE}/etl_entity").collect() ]

for cfg in configs:
    if not cfg["enabled"]:
        run_log.append({"entity_id": cfg["entity_id"], "status": "SKIPPED_DISABLED"}); continue
    t0 = time.time()
    try:
        if cfg["engine_hint"] == "python":
            # single-node path: delta-rs copy, no Spark job at all
            from deltalake import DeltaTable as _DT, write_deltalake as _wd
            data = _DT(cfg["source_path"]).to_pyarrow_table()
            _wd(cfg["target_path"], data, mode="overwrite")
            rows = data.num_rows
        else:
            df = spark.read.format("delta").load(cfg["source_path"])
            writer = df.write.format("delta").mode("overwrite")
            for k, v in cfg["table_props"].items():
                writer = writer.option(k, v)
            writer.save(cfg["target_path"])
            rows = df.count()
        run_log.append({"entity_id": cfg["entity_id"], "status": "OK",
                         "rows": rows, "secs": round(time.time()-t0, 2),
                         "app_id": spark.sparkContext.applicationId})
    except Exception as ex:
        run_log.append({"entity_id": cfg["entity_id"], "status": "FAILED", "error": str(ex)[:200]})

spark.createDataFrame([(str(r),) for r in run_log], ["entry"]) \
     .write.format("delta").mode("append").save(f"{WAREHOUSE}/etl_run_log")
for r in run_log: print(r)
assert all(r["status"] != "FAILED" for r in run_log)

{'entity_id': 1, 'status': 'OK', 'rows': 1000000, 'secs': 7.84, 'app_id': 'local-1785672811472'}
{'entity_id': 3, 'status': 'SKIPPED_DISABLED'}
{'entity_id': 2, 'status': 'OK', 'rows': 12, 'secs': 0.02, 'app_id': 'local-1785672811472'}


---
## 5 — Orchestration: one session, not N

A pipeline invoking 20 notebooks starts (and bills) up to 20 Spark sessions.
`notebookutils.notebook.runMultiple()` runs a **DAG of notebooks inside one shared session** —
the natural executor for the metadata loop above, and typically the single biggest CU saving
available in orchestration. Pipelines remain the right *outer* trigger (schedules, copy
activities, alerts) invoking one coordinator notebook; Activator replaces the clock with events
(file arrival, capacity signals); Spark Job Definitions harden non-interactive batch;
Materialized Lake Views remove orchestration entirely where the transform is SQL.

In [19]:
RUN_MULTIPLE_EXAMPLE = """
dag = {
  "activities": [
    {"name": "bronze_orders",  "path": "nb_ingest_generic", "timeoutPerCellInSeconds": 900,
      "args": {"entity_id": 1}},
    {"name": "bronze_regions", "path": "nb_ingest_generic", "args": {"entity_id": 2}},
    {"name": "silver_orders",  "path": "nb_silver_conform", "args": {"entity_id": 10},
      "dependencies": ["bronze_orders", "bronze_regions"]},
    {"name": "gold_daily",     "path": "nb_gold_build",     "args": {"entity_id": 20},
      "dependencies": ["silver_orders"]}
  ],
  "concurrency": 2
}
notebookutils.notebook.runMultiple(dag)
"""
try:
    import notebookutils
    print("Fabric detected - the DAG below would execute in THIS session:")
except ImportError:
    print("Not in Fabric - reference DAG (paste into a Fabric notebook):")
print(RUN_MULTIPLE_EXAMPLE)

Not in Fabric - reference DAG (paste into a Fabric notebook):

dag = {
  "activities": [
    {"name": "bronze_orders",  "path": "nb_ingest_generic", "timeoutPerCellInSeconds": 900,
      "args": {"entity_id": 1}},
    {"name": "bronze_regions", "path": "nb_ingest_generic", "args": {"entity_id": 2}},
    {"name": "silver_orders",  "path": "nb_silver_conform", "args": {"entity_id": 10},
      "dependencies": ["bronze_orders", "bronze_regions"]},
    {"name": "gold_daily",     "path": "nb_gold_build",     "args": {"entity_id": 20},
      "dependencies": ["silver_orders"]}
  ],
  "concurrency": 2
}
notebookutils.notebook.runMultiple(dag)



---
## 6 — Fabric-only features: ready-to-paste cells

These cannot execute locally (they need Fabric services), so they ship as verified-syntax
reference cells with the guard pattern already applied.

**6.1 — V-Order for a gold table** (Spark write path only — the reason `choose_engine` forces
Spark for gold):
```python
spark.conf.set("spark.sql.parquet.vorder.default", "true")   # check current default first!
(df.write.format("delta").mode("overwrite")
   .option("parquet.vorder.default", "true")
   .save("Tables/gold_sales"))
```

**6.2 — Materialized Lake View** (silver→gold without orchestration; Fabric preview):
```sql
CREATE MATERIALIZED LAKE VIEW gold.daily_revenue
AS SELECT order_date, sum(amount) AS revenue, count(*) AS orders
   FROM silver.orders_silver
   WHERE status = 'complete'
   GROUP BY order_date;
```

**6.3 — Diagnostic emitter to Log Analytics** (Environment Spark properties; §19 of the HTML doc):
```
spark.synapse.diagnostic.emitters: LA
spark.synapse.diagnostic.emitter.LA.type: AzureLogAnalytics
spark.synapse.diagnostic.emitter.LA.categories: Log,EventLog,Metrics
spark.synapse.diagnostic.emitter.LA.workspaceId: <log-analytics-workspace-id>
spark.synapse.diagnostic.emitter.LA.secret.keyVault: <key-vault-uri>
spark.synapse.diagnostic.emitter.LA.secret.keyVault.secretName: <secret-name>
```
Then standing KQL alerts on `OutOfMemoryError`, `FetchFailedException`, and 430 admission
failures turn §19's whole troubleshooting table into proactive signals.

**Closing rule of thumb:** configuration over code (runtime profiles), metadata over hard-coding
(§4), the smallest engine that fits (§2), table properties set at creation (§3), one session for
many notebooks (§5), and logs shipped somewhere queryable before you need them (6.3).

In [20]:
# End-of-notebook verification: everything this notebook created, in one place.
import os
created = [p for p in os.listdir(WAREHOUSE)] if os.path.isdir(WAREHOUSE) else []
print("Artifacts under", WAREHOUSE, "->", sorted(created))
spark.stop()
print("Session stopped cleanly.")

Artifacts under /tmp/fabric_bp_warehouse -> ['bronze_orders', 'bronze_regions', 'etl_entity', 'etl_run_log', 'orders', 'orders_clustered', 'orders_silver', 'region_dim']


Session stopped cleanly.
